In [13]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
PINECONE_API_KEY = os.environ['PINECONE_API_KEY']

In [14]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("battery-test1")
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [20]:
from openai import OpenAI
client =  OpenAI(api_key=OPENAI_API_KEY)

In [21]:
batch_size = 100
df = pd.read_csv("식품영양성분정보_최종.csv")
for start in range(0, len(df), batch_size):
    end = start + batch_size
    batch = df.iloc[start:end]

    texts = [
        " | ".join([f"{col}: {row[col]}" for col in df.columns if pd.notnull(row[col])])
        for _, row in batch.iterrows()
    ]
    embs = client.embeddings.create(model="text-embedding-3-small", input=texts)

    vectors = []
    for j, (_, row) in enumerate(batch.iterrows()):
        vector = embs.data[j].embedding
        metadata = {col: str(row[col]) for col in df.columns if pd.notnull(row[col])}
        metadata["text"] = texts[j]
        food_id = str(row["food_id"]) if pd.notnull(row["food_id"]) else f"row-{start+j}"
        vectors.append((food_id, vector, metadata))

    index.upsert(vectors, namespace="food-data")

print("Pinecone 저장 완료")


Pinecone 저장 완료


In [53]:
SYSTEM_PROMPT = """
You are a professional nutritionist and meal planning expert. 
Your primary responsibility is to create realistic, evidence-based weekly meal plans strictly based on the food and nutrition data provided from the Pinecone vector database.

Key rules:
- Never invent or fabricate information. Only use the context retrieved from the database. 
- Tailor your recommendations according to the user’s specific dietary needs (e.g., weight control, blood sugar management, or low-sodium diets). 
- Each day must include three meals: Breakfast, Lunch, and Dinner. 
- Lunch and Dinner must always include at least one main dish (main menu item). 
- Do not create a meal with only one small side dish (e.g., only “kkakdugi” or a single minor food). Each meal must include at least two items to ensure it is realistic and balanced. 
- Only include foods that exist in the Pinecone vector database. Do not recommend any food that is not retrieved from the database. 
- Each meal must include all relevant nutrition fields: Food name(s), Serving size (g orml), Calories (kcal), Protein (g), Fat (g), Carbohydrates (g), Sugars (g), Sodium (mg), Cholesterol (mg), Saturated Fat (g), and Trans Fat (g).
- The same food item must never appear more than three times across the entire 7-day meal plan. Absolutely avoid using any food more than three times. Ensure strong variety across the week. 
- Provide a complete weekly meal plan (7 days). Present it in a structured table format for each day and each meal. 
- The plan must be realistic, nutritionally balanced, and feasible in real life.
- Always present the 7-day meal plan in a clean markdown table format. 
- Each table must have clear columns such as: Day, Meal (Breakfast/Lunch/Dinner), Food name(s), Serving size (g or ml), Calories (kcal), Protein (g), Fat (g), Carbohydrates (g), Sugars (g).
- Do not use HTML tags such as <br>. Use only markdown formatting for tables and plain text for summaries.

At the end of the weekly meal plan, always include a concise summary section in bullet points. 
The summary must explain:
- The general nutritional strategy of the plan (e.g., focus on protein-rich foods, reduced fat, balanced carbs). 
- How the calorie target aligns with the user’s goals. 
- How food variety was ensured. 
- How balance was achieved in each meal (e.g., inclusion of both staple and protein-rich items). 
- Any special considerations applied (e.g., low sodium, blood sugar control).

Your tone should be professional, supportive, and approachable, helping users follow a sustainable diet that meets their goals without unrealistic restrictions.
Answer in Korean.
"""


In [54]:
from langchain.prompts import ChatPromptTemplate
from openai import OpenAI
import pandas as pd

client = OpenAI(api_key=OPENAI_API_KEY)


# 2. 템플릿 정의 (role을 'user'로 지정!)
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("user", "질문: {question}\n\n관련 데이터:\n{context}")
    ]
)

def rag_pipeline(question: str, top_k: int = 5):
    # (1) 질문 임베딩
    q_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    # (2) Pinecone 검색
    results = index.query(
        vector=q_emb,
        top_k=top_k,
        include_metadata=True,
        namespace="food-data"
    )

    # (3) context 구성
    context = "\n".join([match["metadata"]["text"] for match in results["matches"]])

    # (4) LangChain 템플릿 적용
    prompt = chat_template.format_messages(question=question, context=context)

    # (5) OpenAI Chat 호출 (role 변환 처리!)
    messages = []
    for m in prompt:
        if m.type == "human":
            role = "user"
        elif m.type == "ai":
            role = "assistant"
        else:
            role = m.type
        messages.append({"role": role, "content": m.content})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    # (6) 텍스트 답변
    answer_text = response.choices[0].message.content

    # (7) DataFrame 만들기 (파싱 예시)
    plan = []
    for line in answer_text.split("\n"):
        if any(keyword in line.lower() for keyword in ["day", "breakfast", "lunch", "dinner"]):
            plan.append(line.strip())

    df = pd.DataFrame(plan, columns=["Meal Plan (Raw)"])

    return answer_text, df

In [55]:
answer_text, df_answer = rag_pipeline('''나는 44세 여성이고 현재 60kg인데 55kg까지 감량하고 싶어. 
    어제 저녁에 삼겹살과 소주 2병을 마셔서 칼로리를 초과했어. 
    이걸 보정할 수 있도록 일주일 치 균형 잡힌 식단을 다시 짜줘. 
    조건: 하루 1500~1700kcal, 아침/점심/저녁 3끼 표 형식,
    단백질 충분히, 과일/채소 포함, 군것질·야식·술 제외, 
    회식으로 인한 초과분을 어떻게 보정했는지도 설명해줘.''', top_k=10)

print("=== 답변 ===")
print(answer_text)

=== 답변 ===
다음은 44세 여성의 체중 감량 목표(60kg에서 55kg로)를 위한 일주일 치 균형 잡힌 식단입니다. 하루 칼로리 섭취 목표는 1500~1700kcal로 설정하였으며, 단백질이 충분하고 과일 및 채소를 포함하였습니다. 아래의 표를 참고해 주세요.

| Day         | Meal      | Food name                                   | Serving size | Calories (kcal) | Protein (g) | Fat (g) | Carbohydrates (g) | Sugars (g) |
|-------------|-----------|---------------------------------------------|--------------|------------------|--------------|----------|--------------------|-------------|
| Monday      | Breakfast | 칼로리반닭가슴살곤약죽                       | 100g         | 38               | 2.31         | 0.0      | 6.92               | 0.77        |
|             |           | 바나나                                      | 100g         | 89               | 1.1          | 0.3      | 23.0               | 12.2        |
|             | Lunch     | 삼겹살볶음_낙지                              | 100ml        | 235              | 11.3         | 16.18    | 10.03              | 5.57        |
|             |           | 

In [25]:
query = "단백질이 높은 음식"
query_emb = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

In [26]:
stats = index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'food-data': {'vector_count': 69844}},
 'total_vector_count': 69844,
 'vector_type': 'dense'}
